In [1]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session
from google.cloud import storage

# ==================================================
# 1. CONFIGURATION
# ==================================================

PROJECT_ID = "fintech-lakehouse-dev"
REGION = "us-central1"

CATALOG_NAME = "fintech_lakehouse"
CATALOG_ID = "fintech_lakehouse_bucket"

REST_URI = "https://biglake.googleapis.com/iceberg/v1/restcatalog"

WAREHOUSE = (
    f"bl://projects/{PROJECT_ID}/catalogs/{CATALOG_ID}"
)

# ==================================================
# 2. DATAPROC RUNTIME CONFIGURATION
# ==================================================

session_config = Session()

# Spark 4.x runtime
session_config.runtime_config.version = "3.0"

## JVM DEPENDENCY JARS
BUCKET = "fintech_lakehouse_bucket"
core = [
    "spark-sql-kafka-0-10_2.13-4.0.2.jar",
    "spark-token-provider-kafka-0-10_2.13-4.0.2.jar",
    "kafka-clients-3.9.0.jar",
    "commons-pool2-2.12.0.jar",
    "spark-avro_2.13-4.0.2.jar",
]
auth = [b.name for b in storage.Client().list_blobs(BUCKET, prefix="jars/auth/") if b.name.endswith(".jar")]

jars = [f"gs://{BUCKET}/jars/{j}" for j in core] + [f"gs://{BUCKET}/{n}" for n in auth]
session_config.runtime_config.properties["spark.jars"] = ",".join(jars)
print(len(jars), "jars")

# ==================================================
# 3. ICEBERG CATALOG CONFIGURATION
# ==================================================

builder = (
    DataprocSparkSession.builder
    .appName("FintechLakehouse")
    .dataprocSessionConfig(session_config)

    .config("spark.sql.defaultCatalog", CATALOG_NAME)

    # Iceberg REST catalog
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.type",
        "rest"
    )
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.uri",
        REST_URI
    )
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.warehouse",
        WAREHOUSE
    )

    # GCP project
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.header.x-goog-user-project",
        PROJECT_ID
    )

    # Google authentication
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.rest.auth.type",
        "org.apache.iceberg.gcp.auth.GoogleAuthManager"
    )

    # GCS FileIO
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.io-impl",
        "org.apache.iceberg.gcp.gcs.GCSFileIO"
    )

    # Iceberg SQL extensions
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    # Credential vending
    .config(
        f"spark.sql.catalog.{CATALOG_NAME}.header.X-Iceberg-Access-Delegation",
        "vended-credentials"
    )
)

# ==================================================
# 4. CREATE SPARK SESSION
# ==================================================

spark = builder.getOrCreate()

print("Spark session initialized!")
print("Spark version:", spark.version)

41 jars


/usr/local/lib/python3.13/dist-packages/google/cloud/dataproc_spark_connect/session.py:336: UserWarning: Python version mismatch detected: Client is using Python 3.13, but Dataproc runtime 3.0 uses Python 3.12. This mismatch may cause issues with Python UDF (User Defined Function) compatibility. Consider using Python 3.12 for optimal UDF execution.
  dataproc_config: Session = self._get_dataproc_config()


█████████████████████████████████▌                                              

Spark session initialized!
Spark version: 4.0.2


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.avro.functions import from_avro

PROJECT_ID = "fintech-lakehouse-dev"

TOPIC_NAME = "card-transactions-topic"

CATALOG = 'fintech_lakehouse'

BRONZE_SCHEMA = f"{CATALOG}.bronze"

SILVER_SCHEMA = f"{CATALOG}.silver"

SILVER1 = f"{SILVER_SCHEMA}.decoded_transactions"

BRONZE1 = f"{BRONZE_SCHEMA}.raw_transactions"

BUCKET = "fintech_lakehouse_bucket"

CHECKPOINT1 = f"gs://{BUCKET}/checkpoints/bronze1"
CHECKPOINT2 = f"gs://{BUCKET}/checkpoints/silver/decoded_txns"

print (BRONZE1)
print (SILVER1)

print (CHECKPOINT1)
print (CHECKPOINT2)

fintech_lakehouse.bronze.raw_transactions
fintech_lakehouse.silver.decoded_transactions
gs://fintech_lakehouse_bucket/checkpoints/bronze1
gs://fintech_lakehouse_bucket/checkpoints/silver/decoded_txns


DROP TABLES

In [3]:
spark.sql(f"""
DROP TABLE IF EXISTS {BRONZE1}
""")

DataFrame[]

In [4]:
spark.sql(f"""
DROP TABLE IF EXISTS {SILVER1}
""")

DataFrame[]

In [5]:
spark.sql(f"""
DROP TABLE IF EXISTS `fintech-lakehouse-dev.fintech_lakehouse_bucket.gold.customer_txn_features_5min`
""")

DataFrame[]

REMOVE CHECKPOINTS

In [6]:
!gcloud storage rm -r {CHECKPOINT1}

!gcloud storage rm -r {CHECKPOINT2}

!gcloud storage rm -r "gs://fintech_lakehouse_bucket/gold/checkpoint/customer_txn_features_5min"

Streaming output truncated to the last 5000 lines.
Removing gs://fintech_lakehouse_bucket/gold/checkpoint/customer_txn_features_5min/state/0/535/#1794236625772966...
Removing gs://fintech_lakehouse_bucket/gold/checkpoint/customer_txn_features_5min/state/0/536/#1794236626576321...
Removing gs://fintech_lakehouse_bucket/gold/checkpoint/customer_txn_features_5min/state/0/536/1.delta#1794236627197955...
Removing gs://fintech_lakehouse_bucket/gold/checkpoint/customer_txn_features_5min/state/0/537/#1794236627439530...
Removing gs://fintech_lakehouse_bucket/gold/checkpoint/customer_txn_features_5min/state/0/538/#1794236628355805...
Removing gs://fintech_lakehouse_bucket/gold/checkpoint/customer_txn_features_5min/state/0/539/#1794236629231580...
Removing gs://fintech_lakehouse_bucket/gold/checkpoint/customer_txn_features_5min/state/0/537/1.delta#1794236628101483...
Removing gs://fintech_lakehouse_bucket/gold/checkpoint/customer_txn_features_5min/state/0/538/1.delta#1794236628962841...
Removing